# Vector stores and semantic search



In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

## Part I: Basic vector store implementation

This is a basic vector store implementation. The classes and their purpose are defined below:

class `Document`: 
* Holds the document structure, keeping the text and metadata separated and organized.

class `SearchResult`:
* Holds the similarity score and its corresponding document for a given search result.

class `VectorStore`:
* `__init__`: Initializes the embedding model and empty lists to store documents and their embeddings. 
* `add_documents`: Iterates over each document to generate and store its normalized embedding using the embedding model.
* `search`: Encodes the query into a normalized embedding, calculates cosine similarity scores against all stored embeddings using dot product (the embeddings are normalized), sorts results by score in descending order and returns the `top_k` results.

In [9]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        for doc in documents:
            self.documents.append(doc)
            embedding = self.embedding_model.encode(doc.text, normalize_embeddings=True)
            self.embeddings.append(embedding)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode(query, normalize_embeddings=True)

        scores = np.dot(self.embeddings, query_embedding)

        scored_results = [SearchResult(score, doc) for score, doc in zip(scores, self.documents)]

        scored_results.sort(key=lambda x: x.score, reverse=True)

        return scored_results[:top_k]

Load the [Animal Fun Facts Dataset](https://github.com/ekohrt/animal-fun-facts-dataset). This dataset contains fun facts about various animals. The column `text` is used as the document content, while `animal_name`, `source`, `media_link` and `wikipedia_link` are 
treated as metadata.

In [7]:
df = pd.read_csv("data/animal-fun-facts-dataset.csv")

df.head()

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


Drop rows that do not have any text to avoid errors when generating embeddings.

In [13]:
df = df.dropna(subset=["text"])

Create the documents list by iterating over the dataset rows and instantiating a `Document` object for each one, passing the text and metadata accordingly.

In [14]:
documents = []

for _, row in df.iterrows():
    text = row["text"]
    metadata = {
        "animal_name": row["animal_name"],
        "source": row["source"],
        "media_link": row["media_link"],
        "wikipedia_link": row["wikipedia_link"]
    }
    documents.append(Document(text, metadata))

Print the total number of documents. Originally the dataset has 7734 rows, 
but after dropping rows with missing text, 7731 documents remain.

In [15]:
print(len(documents))

7731


Create the embedding model using `SentenceTransformer` with the `all-MiniLM-L6-v2` model, instantiate the `VectorStore` object, and add the documents list to it.

In [16]:
model = SentenceTransformer("all-MiniLM-L6-v2")

vector_store = VectorStore(model)

vector_store.add_documents(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Define a `print_search_results` helper function to display the score, text and metadata for each result. The following 5 queries are used to test the `search` method and were generate by AI.

In [25]:
def print_search_results(results: list[SearchResult]):
    for result in results:
        print(f"Score: {result.score:.4f}")
        print(f"Text: {result.document.text}")
        print(f"Metadata: {result.document.metadata}")
        print("---")

In [26]:
results = vector_store.search("animals that are good swimmers", top_k=5)

print_search_results(results)

Score: 0.7131
Text: White-tail deer are good swimmers
Metadata: {'animal_name': 'white-tail deer', 'source': 'https://a-z-animals.com/animals/white-tail-deer/', 'media_link': nan, 'wikipedia_link': '/wiki/White-tailed_deer'}
---
Score: 0.6891
Text: These cats are expert swimmers.
Metadata: {'animal_name': 'jaguarundi cat', 'source': 'https://a-z-animals.com/animals/jaguarundi-cat/', 'media_link': nan, 'wikipedia_link': '/wiki/Jaguarundi'}
---
Score: 0.6867
Text: They can also swim.
Not only do they move fast on land, but they are excellent swimmers.
Metadata: {'animal_name': 'komodo dragon', 'source': 'https://factanimal.com/komodo-dragon/', 'media_link': nan, 'wikipedia_link': '/wiki/Komodo_dragon'}
---
Score: 0.6816
Text: They are excellent swimmers and can cross great distances and strong ocean currents just to raid neighboring islands where the only available food source is domestic animals.
Metadata: {'animal_name': 'komodo dragon', 'source': 'https://seaworld.org/animals/facts/re

In [27]:
results = vector_store.search("dangerous predators", top_k=5)

print_search_results(results)

Score: 0.6970
Text: Has no real natural predators!
Metadata: {'animal_name': 'buffalo', 'source': 'https://a-z-animals.com/animals/buffalo/', 'media_link': nan, 'wikipedia_link': '/wiki/Bison'}
---
Score: 0.6970
Text: Has no real natural predators!
Metadata: {'animal_name': 'mountain lion', 'source': 'https://a-z-animals.com/animals/mountain-lion/', 'media_link': nan, 'wikipedia_link': '/wiki/Cougar'}
---
Score: 0.6654
Text: A dominant predator in it's environment!
Metadata: {'animal_name': 'brown bear', 'source': 'https://a-z-animals.com/animals/brown-bear/', 'media_link': nan, 'wikipedia_link': '/wiki/Brown_bear'}
---
Score: 0.6632
Text: A very bold and ferocious predator!
Metadata: {'animal_name': 'ermine', 'source': 'https://a-z-animals.com/animals/ermine/', 'media_link': nan, 'wikipedia_link': '/wiki/Stoat'}
---
Score: 0.6511
Text: In the event of a predator, they will act as a group and fight to protect each other.
Metadata: {'animal_name': 'slender-tailed meerkat', 'source': 'ht

In [28]:
results = vector_store.search("animals that live in the ocean", top_k=5)

print_search_results(results)

Score: 0.6498
Text: Smallest cetacean in the ocean
Metadata: {'animal_name': 'vaquita', 'source': 'https://a-z-animals.com/animals/vaquita/', 'media_link': nan, 'wikipedia_link': '/wiki/Vaquita'}
---
Score: 0.6452
Text: White Sharks live in all of the world's oceans.
Metadata: {'animal_name': 'white shark', 'source': 'https://a-z-animals.com/animals/white-shark/', 'media_link': nan, 'wikipedia_link': '/wiki/Great_white_shark'}
---
Score: 0.6317
Text: May eat squid or other small invertebrate ocean life
Metadata: {'animal_name': 'bonito fish', 'source': 'https://a-z-animals.com/animals/bonito-fish/', 'media_link': nan, 'wikipedia_link': '/wiki/Bonito'}
---
Score: 0.6217
Text: Sharks live all over the world, from warm, tropical lagoons to polar seas. Some even inhabit freshwater lakes and rivers!
Metadata: {'animal_name': 'sharks', 'source': 'https://seaworld.org/animals/facts/cartilaginous-fish/sharks/', 'media_link': nan, 'wikipedia_link': '/wiki/Shark'}
---
Score: 0.6217
Text: They br

In [29]:
results = vector_store.search("animals that can fly", top_k=5)

print_search_results(results)

Score: 0.7076
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
Metadata: {'animal_name': 'colugo (flying lemur)', 'source': 'https://factanimal.com/colugo/', 'media_link': nan, 'wikipedia_link': '/wiki/Colugo'}
---
Score: 0.6901
Text: Not all birds are able to fly!
Metadata: {'animal_name': 'bird', 'source': 'https://a-z-animals.com/animals/bird/', 'media_link': nan, 'wikipedia_link': '/wiki/Bird'}
---
Score: 0.6481
Text: They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
Metadata: {'animal_name': 'secretary bird', 'source': 'https://factanimal.com/secretarybird/', 'media_link': nan, 'wikipedia_link': '/wiki/Secretarybird'}
---
Score: 0.6437
Text: Bats are the only mammals with wings, and the only ones that can truly fly
Metadata: {'animal_name': 'bat', 'source': 'https://www.animalfactsencyclopedia.c

In [30]:
results = vector_store.search("largest land animals", top_k=5)

print_search_results(results)

Score: 0.8334
Text: The second largest animal on the land!
Metadata: {'animal_name': 'white rhinoceros', 'source': 'https://a-z-animals.com/animals/white-rhinoceros/', 'media_link': nan, 'wikipedia_link': '/wiki/White_rhinoceros'}
---
Score: 0.8121
Text: The largest animal on Earth
Metadata: {'animal_name': 'blue whale', 'source': 'https://a-z-animals.com/animals/blue-whale/', 'media_link': nan, 'wikipedia_link': '/wiki/Blue_whale'}
---
Score: 0.7434
Text: The buffalo is the largest land animal in the New World
Metadata: {'animal_name': 'buffalo', 'source': 'https://www.animalfactsencyclopedia.com/Buffalo-facts.html', 'media_link': nan, 'wikipedia_link': '/wiki/Bison'}
---
Score: 0.7073
Text: Elephants are the largest living land animals on earth..
The African bull elephant can grow as large as 13 feet (4 meters) tall, weigh between 4,000-7,500 kg and can have tusks as long as 6.5 feet (2 meters) in length weighing 100 pounds each (45 kg).
Metadata: {'animal_name': 'elephant', 'source'

All queries returned semantically related results, even when the exact words did not match. 

## Part II: Filtering by metadata

The class `FilteredVectorStore` contains the following methods:
* `__init__`: Same implementation as `VectorStore`.
* `add_documents`: Same implementation as `VectorStore`.
* `search`: Extends the `VectorStore` search by accepting an optional `metadata_filter` parameter. If provided, it first filters the documents whose metadata matches the filter, then applies semantic search only over that subset. If no filter is provided, it searches over all documents.

In [39]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        for doc in documents:
            self.documents.append(doc)
            embedding = self.embedding_model.encode(doc.text, normalize_embeddings=True)
            self.embeddings.append(embedding)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if metadata_filter is not None:
            filtered_docs = []
            filtered_embeddings = []
            for doc, emb in zip(self.documents, self.embeddings):
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items()):
                    filtered_docs.append(doc)
                    filtered_embeddings.append(emb)
        else:
            filtered_docs = self.documents
            filtered_embeddings = self.embeddings

        query_embedding = self.embedding_model.encode(query, normalize_embeddings=True)

        scores = np.dot(filtered_embeddings, query_embedding)

        scored_results = [SearchResult(score, doc) for score, doc in zip(scores, filtered_docs)]

        scored_results.sort(key=lambda x: x.score, reverse=True)

        return scored_results[:top_k]



The new dataset consists of 2225 news articles from the BBC website, covering stories in five categories from 2004-2005: business, entertainment, politics, sport and tech. The `content` column is used as the document text, while `category`, `filename` and `title` 
are treated as metadata.

Source: [BBC News Archive - Kaggle](https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive)

In [40]:
df = pd.read_csv("data/bbc-news-data.csv", sep="\t")

df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


Same implementation as the Animal Fun Facts dataset to create the documents list, mapping `content` as the document text and `category`, `filename` and `title` as metadata.

In [41]:
documents = []

for _, row in df.iterrows():
    text = row["content"]
    metadata = {
        "category": row["category"],
        "filename": row["filename"],
        "title": row["title"]
    }
    documents.append(Document(text, metadata))

In [42]:
print(len(documents))

2225


Reuse the same embedding model as before and instantiate a `FilteredVectorStore` to allow metadata filtering during search.

In [43]:
filtered_vector_store = FilteredVectorStore(model)

filtered_vector_store.add_documents(documents)

The following 5 queries were created with AI (only the query strings), one per category, using the `metadata_filter` parameter to restrict the search to a specific category.

In [46]:
results = filtered_vector_store.search(
    "stock market profits", 
    top_k=5, 
    metadata_filter={"category": "business"}
)
print_search_results(results)

Score: 0.4976
Text:  Oil giant BP has announced a 26% rise in annual profits to $16.2bn (£8.7bn) on the back of record oil prices.  Last week, rival Shell reported an annual profit of $17.5bn - a record profit for a UK-listed company. BP added that it was increasing its fourth-quarter dividend by 26% to 8.5 cents, and that it would continue with share buybacks. BP chief executive Lord Browne said the results were strong "both operationally and financially."  The company is earning about $1.8m an hour.  Despite the record annual profits figure, BP's performance was below the expectations of some City analysts. However, BP's share price rose 4p or nearly 1% in morning trading to 548p. Its profit rise for the year included profits of $3.65bn (£1.97bn) for the final three months of 2004 - up from $2.89bn a year ago but below its third quarter.  Speaking on the BBC's Today programme on Tuesday, Lord Browne said the profits were not solely down to the high oil price alone.  "The profits are 

In [48]:
results = filtered_vector_store.search(
    "the latest movies", 
    top_k=5, 
    metadata_filter={"category": "entertainment"}
)
print_search_results(results)

Score: 0.4087
Text:  It's a universal rule that a film can either be a superhero special effects extravaganza or it can be good. But Spider-Man 2 breaks that rule in two.  It's not fantastically deep but you get quickly drawn into the tale of Spidey versus Doc Ock and more so into the fate of poor Peter Parker. Gigantic action set pieces seamlessly work with more brooding personal torment and it all looks stunning. A few effects look false but Tobey Maguire, Kirsten Dunst and Alfred Molina make this compelling. The other universal rule is that DVDs of superhero films will have Making Of features only about the effects. This disc covers those special effects enough but as just one part of a detailed look at the film. Then there are commentaries, trailers and a blooper reel.  Sometimes quality comes in bulk: this set contains no less than 34 John Wayne films ranging from the Westerns and war movies to The Quiet Man.  Now is that a Christmas present or what? Give this to someone on 24 Dec

In [50]:
results = filtered_vector_store.search(
    "election voting results", 
    top_k=5, 
    metadata_filter={"category": "politics"}
)
print_search_results(results)

Score: 0.4298
Text:  The Lib Dems are set for their best results in both the general election and the local council polls, one of their frontbenchers has predicted.  Local government spokesman Ed Davey was speaking as the party launched its campaign for the local elections being held in 37 English council areas. The flagship pledge is to replace council tax with a local income tax. The Tories say the Lib Dems would make people pay more tax and Labour says the party's sums do not add up. Looking to the coming elections, which are all expected to be held on 5 May, Mr Davey said: "We are going to be winning more votes and winning more seats. "I think we are going to have the best general election results and local election results we have ever had under [party leader] Charles Kennedy. "I couldn't think of a stronger endorsement of a leader." 
Metadata: {'category': 'politics', 'filename': '398.txt', 'title': "Lib Dems predict 'best ever poll'"}
---
Score: 0.3862
Text:  Three Labour counci

In [51]:
results = filtered_vector_store.search(
    "football world cup", 
    top_k=5, 
    metadata_filter={"category": "sport"}
)
print_search_results(results)

Score: 0.4487
Text:  Hong Kong is hoping to join Japan as co-host of the 2011 Rugby World Cup.  Japan has applied to host the tournament on its own, with the aim of taking it outside rugby's traditional strongholds for the first time. But Hong Kong Rugby Football Union (HKRFU) chairman John Molloy has called for the territory to host one of the pools and a quarter-final. The Japanese Rugby Football Union (JRFU) says it has yet to receive a formal presentation from the HKRFU. "At this stage, we are only considering hosting the event by ourselves," said JRFU secretary Koji Tokumasu. "We cannot examine any proposal unless we get it in a definitive form." Japan faces stiff competition in the form of South Africa and New Zealand to host the event in seven years' time.  "Until now, the World Cup has been held in countries from the Six Nations or Tri-Nations," said Tokumasu. "We think, and the IRB thinks, that it is time for rugby to go global. "Japan is ready to host the tournament and we ar

In [53]:
results = filtered_vector_store.search(
    "artificial intelligence trends", 
    top_k=5, 
    metadata_filter={"category": "tech"}
)
print_search_results(results)

Score: 0.3681
Text:  Robots are learning lessons on "robotiquette" - how to behave socially - so they can mix better with humans.  By playing games, like pass-the-parcel, a University of Hertfordshire team is finding out how future robot companions should react in social situations. The study's findings will eventually help humans develop a code of social behaviour in human-robot interaction. The work is part of the European Cogniron robotics project, and was on show at London's Science Museum.  "We are assuming a situation in which a useful human companion robot already exists," said Professor Kerstin Dautenhahn, project leader at Hertfordshire. "Our mission is to look at how such a robot should be programmed to respect personal spaces of humans."  The research also focuses on human perception of robots, including how they should look, and how a robot can learn new skills by imitating a human demonstrator. "Without such studies, you will build robots which might not respect the fact t

The results of the semantic search are consistent and relevant. All returned documents are related to the query and belong to the filtered category, confirming that both the semantic search and the metadata filtering are working correctly.

## Final Insights

This activity allowed me to understand how vector stores and semantic search work in practice. In the basic implementation, I was able to create documents and perform semantic search over them. The results were relevant, for example, querying for flying animals returned documents related to that topic with good similarity scores.

In the `FilteredVectorStore`, the only addition was the metadata filter, which restricts the search to a subset of documents before applying semantic search. This is effective when looking for information within a specific category, avoiding unnecessary comparisons with unrelated documents.

It is also worth noting that the similarity scores varied between the two datasets. The Animal Fun Facts dataset contains short sentences, making it easier to find strong semantic matches. The BBC News dataset contains full articles, which makes semantic similarity harder to capture in a single embedding, resulting in lower scores overall.